# NB-17 — variance components and the run-clustered ANCOVA

**CPU only. Runs in under a minute. No model is loaded.** Set the accelerator to None; this
does not need a GPU and should not spend GPU quota.

## What it does

1. **Reproduces the paper's two ANCOVA fits** ($+0.1517$ over six conditions, $+0.1007$ over
   seven) from the attached CSVs, and refuses to be trusted if they do not come back.
2. **Refits clustered by training run.** Once a condition has more than one run, the condition
   is no longer the independent unit — the run is. Clustering by condition while pooling
   replicates inside it treats two draws as one and understates the standard error.
3. **Estimates between-run sigma**, pooled within replicate groups, bias-corrected, with a
   chi-square interval. It reproduces the paper's "about 30 points" from the two existing
   1-hour runs, so the estimator is verified against a published number before it produces a
   new one.
4. **Turns sigma into runs per condition**, with the interval — the number `sec:power`
   currently quotes from an $n=2$ extrapolation.

## Why sigma has two values, and which to report

The sample standard deviation is badly biased low at tiny $n$: $E[s] = c_4(\mathrm{df})\,\sigma$,
and $c_4(1) = 0.7979$. From the two 1-hour runs the raw pooled sd is $34/\sqrt{2} = 24.04$, and
the bias-corrected value is $24.04/0.7979 = 30.1$ — exactly the figure the paper reports,
arrived at there by inverting the expected range ($E[\mathrm{range}] = 1.128\sigma$ for $n=2$).
**The corrected value is the one to report.** The notebook prints both, because printing only
the raw one would make the manuscript look inconsistent with its own analysis.

The correction stops mattering as replicates arrive: $c_4 = 0.798, 0.886, 0.921, 0.940, 0.952$
at df $1 \ldots 5$.

## Attach

Every `eps50-*-n100` dataset — the six Task B runs, `eps50-rung1-trajfinal-t5-n100`, and the
NB-16 replicate output — plus `clean-wer-replicates` from NB-15.

A replicate that has been attacked but has no measured clean WER is **skipped with a printed
warning**, not silently given its published sibling's WER. Substituting it would conceal the
very thing being measured.


## 1 · Configuration and the published reference values

In [ ]:
# NB-17 runs on CPU in minutes. It reads CSVs and writes tables; no model is loaded.
#
# LOCAL_DIR lets the same notebook run off Kaggle -- point it at a folder holding
# the eps50_*.csv files and clean_wer_replicates.csv. On Kaggle leave it None.
LOCAL_DIR = None

EDIT_TYPE = "T5"
N_EVAL    = 100
CEIL      = 2e-3          # the eps50 ceiling; censored rows are imputed here
ALPHA     = 0.05
POWER     = 0.80

import os, re, json, glob, time, math, itertools
import numpy as np, pandas as pd

# ---------------------------------------------------------------------------
# THE PUBLISHED CONDITIONS.
#
# `hours` is EXPOSURE -- distinct training audio the checkpoint has seen -- not
# the rung label. For a milestone of the 100 h run it is (distinct clips seen /
# 80,773) x 98.02 h, which is why ms-1000 is 38.8 h and not 98 h. These are the
# numbers behind the "40x / 16x / 4x" exposure ratios in the pairs table.
#
# `wer` is clean WER on the SAME 100 utterances the attack is measured on
# (paper Table 2 / tab:eps50). Not the 25-utterance figures, and not dev WER.
#
# Source: paper/main.tex tab:eps50 + tab:iso; RUNLOG 2026-09-30 (NB-02b) and
# 2026-10-07 (NB-02c).
# ---------------------------------------------------------------------------
# `expect_asr` is what the paper reports for each condition. It is checked after
# loading, because of a hazard documented at length in RUNLOG:
#
#   `eps50-rung1-final-t5-n100` (Task B run 1, ASR 75%) and
#   `eps50-rung1-traj-final-t5-n100` (Task B run 7, the SAME traj model, ASR 61%)
#   BOTH contain a file called `eps50_rung1_final_T5_n100.csv`.
#
# Attach both and a basename-keyed lookup silently picks one. The condition then
# carries the other model's numbers under the right label. The ASR check below
# catches that in seconds.
PUBLISHED = {
    # key                        hours   clean WER   rung   file
    "1h final":        dict(hours=0.98,  wer=0.6355, rung=1,  group="rung1-final", expect_asr=75.0, trun="1h-r1",
                            file=f"eps50_rung1_final_{EDIT_TYPE}_n{N_EVAL}.csv"),
    "5h final":        dict(hours=4.90,  wer=0.4233, rung=5,  group="rung5-final", expect_asr=79.0, trun="5h-r1",
                            file=f"eps50_rung5_final_{EDIT_TYPE}_n{N_EVAL}.csv"),
    "25h final":       dict(hours=24.60, wer=0.3114, rung=25, group="rung25-final", expect_asr=66.0, trun="25h-r1",
                            file=f"eps50_rung25_final_{EDIT_TYPE}_n{N_EVAL}.csv"),
    "100h ms-1000":    dict(hours=38.80, wer=0.6943, rung=100, group=None, expect_asr=37.0, trun="100h-r1",
                            file=f"eps50_rung100_milestone-1000_{EDIT_TYPE}_n{N_EVAL}.csv"),
    "100h ms-2000":    dict(hours=77.70, wer=0.4752, rung=100, group=None, expect_asr=46.0, trun="100h-r1",
                            file=f"eps50_rung100_milestone-2000_{EDIT_TYPE}_n{N_EVAL}.csv"),
    "100h ms-3500":    dict(hours=98.02, wer=0.3449, rung=100, group=None, expect_asr=53.0, trun="100h-r1",
                            file=f"eps50_rung100_milestone-3500_{EDIT_TYPE}_n{N_EVAL}.csv"),
    # the paper's second 1 h training run -- same seed, divergent trajectory
    "1h run2":         dict(hours=0.98,  wer=0.6251, rung=1,  group="rung1-final", expect_asr=41.0, trun="1h-r2",
                            file=f"eps50_rung1-trajfinal_{EDIT_TYPE}_n{N_EVAL}.csv"),
}

# `trun` is the TRAINING RUN that produced the checkpoint, and it is the cluster.
#
# It is NOT the same as the condition. The three 100-hour milestones are three
# CHECKPOINTS OF ONE TRAINING RUN -- ms-1000, ms-2000 and ms-3500 are step
# 1000, 2000 and 3500 of the same trajectory, scored on the same 100 utterances.
# CR1 assumes independence ACROSS clusters, and those three are not independent.
#
# Clustering on the 13 condition labels therefore counts 13 independent units
# where there are 11, and understates the standard error of beta1. The fit
# clustered on `trun` is the defensible one; the 13-label fit is reported beside
# it so the difference is visible rather than a choice the reader must trust.
#
# `group` is the REPLICATE group: two runs are replicates of each other only if
# they share it. It is NOT the same as sharing an exposure.
#
# ms-3500 and a 100 h final both have exposure 98.02 h -- every clip seen -- but
# one stopped at step 3500 and the other at 6000, so they differ by 2500 steps of
# optimisation. Pooling them as "two runs at 98 h" would charge that real
# difference to run-to-run noise and inflate sigma. Milestones therefore carry
# group=None and never enter the variance estimate. Only finals of the same rung
# are replicates.

# The two fits the paper reports, which this notebook must reproduce before any
# new number from it is trusted. Source: paper tab:ancova.
SIX   = ["1h final", "5h final", "25h final",
         "100h ms-1000", "100h ms-2000", "100h ms-3500"]
SEVEN = SIX + ["1h run2"]
EXPECT = {6: (+0.1517, (+0.080, +0.224)), 7: (+0.1007, (-0.006, +0.208))}

# Rung label -> exposure hours, for replicates. A replicate of the 5 h rung has
# the 5 h rung's exposure by construction: same manifest, same clips.
RUNG_HOURS = {1: 0.98, 5: 4.90, 25: 24.60, 100: 98.02}

# The iso-WER pair effects the variance has to be read against (paper tab:iso).
PAIR_EFFECTS = {1: ("40x", 38.0), 5: ("16x", 33.0), 25: ("4x", 13.0)}

# ---------------------------------------------------------------------------
# SOFTWARE ENVIRONMENT, and why it is a grouping variable here.
#
# The published ladder trained on transformers 4.44.2. Kaggle now ships 5.16.1,
# and NB-01's pip pin cannot be restored with internet off. So a replicate
# compared against its published sibling differs in TWO things -- the training
# run and the library version -- and a sigma pooled across that comparison
# conflates them. A reviewer would be right to say so.
#
# The primary sigma is therefore pooled WITHIN (replicate group, environment):
# the s1338-vs-s1339 pair at each rung, both on the current library, only the
# run differing. The all-runs sigma is reported too, labelled as confounded.
#
# If Kaggle moves again mid-way through, set REPLICATE_ENV from the actual log
# lines and add an entry to ENV_OVERRIDE for any run that differs.
# ---------------------------------------------------------------------------
PUBLISHED_ENV = "tf4.44.2"     # ckpt-xlsr-{1,5,25,100}h and -1h-traj
REPLICATE_ENV = "tf5.16.1"     # every ckpt-xlsr-*h-s* built by NB-14
ENV_OVERRIDE  = {}             # e.g. {"5h s1339": "tf5.17.0"}

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
print(f"NB-17 | variance components + run-clustered ANCOVA | writing to {WORK}")

## 2 · Find the inputs

Searches `/kaggle/working` and every `/kaggle/input` depth, or `LOCAL_DIR` if set.

In [ ]:
def _search_roots():
    roots = []
    if LOCAL_DIR:
        roots.append(LOCAL_DIR)
    roots += ["/kaggle/working", "/kaggle/input/*", "/kaggle/input/*/*",
              "/kaggle/input/*/*/*"]
    return roots

# ---------------------------------------------------------------------------
# Discovery is keyed on BASENAME, and two different datasets can carry the same
# basename. That is not hypothetical here: Task B named every run's output
# `eps50_rung{RUNG}_{CHECKPOINT}_{EDIT_TYPE}_n100.csv` from the rung and the
# checkpoint, so run 1 (ckpt-xlsr-1h/final, ASR 75%) and run 7
# (ckpt-xlsr-1h-traj/final, ASR 61%) BOTH produced
# `eps50_rung1_final_T5_n100.csv`.
#
# A `setdefault` would take whichever mounted first and the analysis would never
# know. So every path is kept, and a basename with more than one path is a hard
# stop that prints the candidates and their ASRs.
# ---------------------------------------------------------------------------
_paths = {}
for root in _search_roots():
    for p in glob.glob(os.path.join(root, "eps50_*.csv")):
        _paths.setdefault(os.path.basename(p), [])
        if p not in _paths[os.path.basename(p)]:
            _paths[os.path.basename(p)].append(p)

def _outcomes(path):
    """utt_id -> success bool. The identity of a measurement, row order aside."""
    d = pd.read_csv(path, usecols=["utt_id", "censored"])
    c = d.censored.astype(str).str.lower().isin(("true", "1", "yes"))
    return dict(zip(d.utt_id.astype(str), ~c))

def _asr(m):
    return 100.0 * sum(m.values()) / len(m) if m else float("nan")

# ---------------------------------------------------------------------------
# Two kinds of duplicate, and they need opposite treatment.
#
#  1. A RESUME CHAIN. A partial session wrote 80 rows; the continuation
#     resumed from it and wrote 100, the first 80 of which came from the
#     partial itself. Both are attached because the earlier notebook also
#     holds a file nothing else does. One set of utterances is a strict
#     SUBSET of the other and they agree on every shared utterance.
#     -> not an ambiguity. Take the larger and say so.
#
#  2. A GENUINE CONFLICT. Task B named outputs from the rung and the
#     checkpoint, so run 1 (ckpt-xlsr-1h/final, 75%) and run 7
#     (ckpt-xlsr-1h-traj/final, 61%) BOTH wrote
#     eps50_rung1_final_T5_n100.csv. Same utterances, different model,
#     different outcomes.
#     -> stop. Picking one silently would put another model's numbers under
#        this condition's label.
#
# The test distinguishes them exactly: subset plus agreement on the overlap.
# ---------------------------------------------------------------------------
_conflicts = []
for _b, _v in sorted(_paths.items()):
    if len(_v) == 1:
        continue
    try:
        _m = {q: _outcomes(q) for q in _v}
    except Exception as e:
        _conflicts.append((_b, _v, f"could not read one of them: {type(e).__name__}: {e}"))
        continue
    _order = sorted(_v, key=lambda q: -len(_m[q]))
    _best = _order[0]
    _why = None
    for q in _order[1:]:
        if not set(_m[q]) <= set(_m[_best]):
            _why = (f"{os.path.basename(os.path.dirname(q))} has utterances that "
                    f"{os.path.basename(os.path.dirname(_best))} does not")
            break
        _bad = [u for u in _m[q] if _m[_best][u] != _m[q][u]]
        if _bad:
            _why = (f"they disagree on {len(_bad)} shared utterance(s), e.g. {_bad[0]}")
            break
    if _why:
        _conflicts.append((_b, _v, _why))
    else:
        print(f"resume chain for {_b}:")
        for q in _order:
            print(f"   n={len(_m[q]):3d}  ASR {_asr(_m[q]):5.1f}%  "
                  f"{'USING' if q == _best else 'subset, ignored'}   {q}")
        _paths[_b] = [_best]

if _conflicts:
    print("\n*** SAME FILENAME, DIFFERENT MEASUREMENTS ***\n")
    for _b, _v, _why in _conflicts:
        print(f"  {_b}  --  {_why}")
        for q in _v:
            try:
                _mm = _outcomes(q)
                print(f"     n={len(_mm):3d}  ASR {_asr(_mm):5.1f}%   {q}")
            except Exception:
                print(f"     unreadable   {q}")
    raise AssertionError(
        "Two attached sources carry the same eps50 filename and are NOT the same "
        "measurement, nor a partial and its continuation -- see above.\n\n"
        "The known case is the Task B naming collision: run 1 "
        "(ckpt-xlsr-1h/final, 75%) and run 7 (ckpt-xlsr-1h-traj/final, 61%) both "
        "wrote eps50_rung1_final_T5_n100.csv.\n\n"
        "FIX: detach one. For the 1 h rung keep `eps50-rung1-final-t5-n100` "
        "(run 1) and DETACH `eps50-rung1-traj-final-t5-n100` (run 7). The traj "
        "measurement this notebook wants is NB-12's, under the distinct basename "
        "eps50_rung1-trajfinal_T5_n100.csv.")

FOUND = {k: v[0] for k, v in _paths.items()}

# All clean-WER tables, concatenated. NB-15 may have been run more than once
# (ten checkpoints is a lot of mount), and taking only the first file would
# silently drop half the replicates from the fit.
_cleans = []
for root in _search_roots():
    for p in glob.glob(os.path.join(root, "clean_wer_replicates.csv")):
        if p not in _cleans:
            _cleans.append(p)
CLEAN = _cleans[0] if _cleans else None

print(f"{len(FOUND)} eps50 CSVs visible:")
for k in sorted(FOUND):
    print("   ", k)
print(f"clean_wer_replicates.csv: {len(_cleans)} file(s)"
      + (f" -> {_cleans}" if _cleans else " -- NOT FOUND, replicates cannot enter the fit"))
assert FOUND, ("no eps50_*.csv found. Attach the published eps50-*-n100 datasets "
               "and this session's NB-16 output, or set LOCAL_DIR.")

## 3 · Build the run table

One row per attacked utterance, one record per training run. Censored rows are ceiling-imputed at 2e-3, which is conservative: it understates how hard a censored utterance really was.

In [ ]:
def load_rows(path, label, hours, rung):
    """One eps50 CSV -> tidy utterance rows. Censored rows are ceiling-imputed."""
    d = pd.read_csv(path)
    for c in ("utt_id", "censored", "eps50"):
        assert c in d.columns, f"{os.path.basename(path)} has no '{c}' column"
    cens = d.censored.astype(str).str.lower().isin(("true", "1", "yes"))
    eps = np.where(cens, CEIL, pd.to_numeric(d.eps50, errors="coerce"))
    # A non-censored row with a missing eps50 is a corrupt file, not a success.
    bad = int(np.sum(~cens & ~np.isfinite(eps)))
    assert bad == 0, (f"{os.path.basename(path)}: {bad} rows are marked NOT censored "
                      "but carry no eps50. Do not impute these -- the file is wrong.")
    return pd.DataFrame(dict(utt_id=d.utt_id.astype(str), run=label, rung=rung,
                             hours=hours, eps50=eps, censored=cens.values,
                             success=~cens.values))

ROWS, RUNMETA, MISSING = [], [], []

# --- the published conditions -------------------------------------------------
for key, m in PUBLISHED.items():
    if m["file"] not in FOUND:
        MISSING.append(key); continue
    r = load_rows(FOUND[m["file"]], key, m["hours"], m["rung"])
    ROWS.append(r)
    RUNMETA.append(dict(run=key, rung=m["rung"], hours=m["hours"], wer=m["wer"],
                        group=m["group"], env=ENV_OVERRIDE.get(key, PUBLISHED_ENV),
                        trun=m["trun"], n=len(r), asr=100.0 * r.success.mean(),
                        source="published"))

# --- the replicates -----------------------------------------------------------
# A replicate needs BOTH its eps50 CSV and its measured clean WER. Without the
# WER it cannot enter an ANCOVA that uses WER as a covariate, and substituting
# its published sibling's WER would hide the very thing being measured.
cw = (pd.concat([pd.read_csv(x) for x in _cleans], ignore_index=True)
            .drop_duplicates(subset="model", keep="last")
      if _cleans else pd.DataFrame())
if len(_cleans) > 1:
    print(f"merged {len(_cleans)} clean-WER tables -> {len(cw)} distinct checkpoints")
for fn, path in sorted(FOUND.items()):
    m = re.fullmatch(rf"eps50_rung(\d+)-s(\d+)_{EDIT_TYPE}_n{N_EVAL}\.csv", fn)
    if not m:
        continue
    rung, seed = int(m.group(1)), int(m.group(2))
    label = f"{rung}h s{seed}"
    hit = cw[cw.get("model", pd.Series(dtype=str)) == f"ckpt-xlsr-{rung}h-s{seed}"] \
          if len(cw) else cw
    if not len(hit):
        print(f"  SKIPPING {label}: attacked, but no clean WER in "
              "clean_wer_replicates.csv. Run NB-15 on this checkpoint.")
        continue
    wer = float(hit.wer_attack100.iloc[0])
    r = load_rows(path, label, RUNG_HOURS[rung], rung)
    ROWS.append(r)
    RUNMETA.append(dict(run=label, rung=rung, hours=RUNG_HOURS[rung], wer=wer,
                        group=f"rung{rung}-final",
                        env=ENV_OVERRIDE.get(label, REPLICATE_ENV),
                        trun=f"{rung}h-s{seed}", n=len(r),
                        asr=100.0 * r.success.mean(), source="replicate"))

assert ROWS, "nothing loaded"
D  = pd.concat(ROWS, ignore_index=True)
RM = pd.DataFrame(RUNMETA).sort_values(["hours", "run"]).reset_index(drop=True)
RM["group"] = RM["group"].where(RM["group"].notna(), None)
D  = D.merge(RM[["run", "wer", "trun"]], on="run", how="left")
assert D.wer.notna().all()

# ---- does each published condition carry the ASR the paper reports? --------
# A shadowed file (see the collision note in cell 1) shows up here instantly:
# the label would be right and the numbers would belong to another model.
print()
_bad = []
for r in RM.itertuples():
    exp = PUBLISHED.get(r.run, {}).get("expect_asr")
    if exp is None:
        continue
    d = r.asr - exp
    flag = "OK" if abs(d) < 0.75 else "MISMATCH"
    if flag == "MISMATCH":
        _bad.append((r.run, exp, r.asr, FOUND.get(PUBLISHED[r.run]["file"])))
    print(f"  {r.run:14s} paper {exp:5.1f}%  loaded {r.asr:5.1f}%  "
          f"({d:+.1f})  {flag}")
assert not _bad, (
    "A published condition did not load the ASR the paper reports:\n  "
    + "\n  ".join(f"{k}: paper {e}%, loaded {g:.1f}%  from {p}" for k, e, g, p in _bad)
    + "\n\nThe label is right and the numbers are not. Most likely a filename "
      "collision between two attached datasets, or the wrong dataset version. "
      "Do not proceed -- every number below would inherit it.")
print("  all attached published conditions match the paper")

if MISSING:
    print(f"\nPublished conditions NOT attached ({len(MISSING)}): {MISSING}")
    print("  The reproduction check below needs all six of the SIX list.")
print(f"\n{len(RM)} runs, {len(D)} utterance rows")
print(RM.to_string(index=False))

## 4 · The estimator

CR1 cluster-robust OLS with $t$ on $G-1$ degrees of freedom — the project's pre-specified estimator, unchanged. The only new choice is the clustering level.

In [ ]:
# Student-t quantiles for the df we actually use, so no scipy dependency.
# Two-sided 0.975 quantile.
T975 = {1: 12.7062, 2: 4.3027, 3: 3.1824, 4: 2.7764, 5: 2.5706, 6: 2.4469,
        7: 2.3646, 8: 2.3060, 9: 2.2622, 10: 2.2281, 11: 2.2010, 12: 2.1788,
        13: 2.1604, 14: 2.1448, 15: 2.1314, 16: 2.1199, 17: 2.1098, 18: 2.1009,
        19: 2.0930, 20: 2.0860, 25: 2.0595, 30: 2.0423, 40: 2.0211, 60: 2.0003}

def t_crit(df):
    if df in T975:
        return T975[df]
    try:
        from scipy.stats import t as _t
        return float(_t.ppf(0.975, df))
    except Exception:
        ks = sorted(T975)
        return T975[min(ks, key=lambda k: abs(k - df))]

def ancova(d, cluster):
    """OLS of log10(eps50) on log10(hours) + clean WER, CR1 cluster-robust SEs,
    t with df = G-1. This is the project's pre-specified estimator; the only
    choice here is what `cluster` is."""
    y = np.log10(d.eps50.values.astype(float))
    X = np.column_stack([np.ones(len(d)),
                         np.log10(d.hours.values.astype(float)),
                         d.wer.values.astype(float)])
    g = d[cluster].values
    N, K = X.shape
    XtX_inv = np.linalg.inv(X.T @ X)
    beta = XtX_inv @ (X.T @ y)
    u = y - X @ beta
    groups = pd.unique(g)
    G = len(groups)
    assert G > K, (f"{G} clusters for {K} parameters -- the CR1 variance is not "
                   "defined. Attach more conditions.")
    meat = np.zeros((K, K))
    for gg in groups:
        s = g == gg
        Xg, ug = X[s], u[s]
        sc = Xg.T @ ug
        meat += np.outer(sc, sc)
    c = (G / (G - 1.0)) * ((N - 1.0) / (N - K))
    V = XtX_inv @ (c * meat) @ XtX_inv
    se = np.sqrt(np.diag(V))
    df = G - 1
    tc = t_crit(df)
    return dict(beta=beta, se=se, df=df, G=G, N=N,
                ci=[(b - tc * s, b + tc * s) for b, s in zip(beta, se)],
                t=[b / s for b, s in zip(beta, se)])

NAMES = ["constant", "log10(hours)", "clean_WER"]

def show(fit, title):
    print(f"\n  {title}")
    print(f"    G={fit['G']} clusters, N={fit['N']} rows, t df={fit['df']}")
    for nm, b, s, (lo, hi), tt in zip(NAMES, fit["beta"], fit["se"], fit["ci"], fit["t"]):
        print(f"    {nm:14s} {b:+.4f}  se {s:.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  t {tt:+.2f}")
    return fit

## 5 · Reproduce the paper's two fits, then refit by run

**The reproduction check is the gate.** If the 6- and 7-condition fits do not come back at $+0.1517$ and $+0.1007$, the inputs are not the ones the manuscript was written from, and nothing below should be used.

In [ ]:
print("\n" + "=" * 74)
print("REPRODUCTION CHECK  --  the two fits the paper reports")
print("=" * 74)
print("  If these do not reproduce, STOP. Every new number below comes from the")
print("  same estimator on the same data, so a mismatch here invalidates all of it.")

ok = True
for n, keys in ((6, SIX), (7, SEVEN)):
    have = [k for k in keys if k in set(RM.run)]
    if len(have) != len(keys):
        print(f"\n  {n}-condition fit: only {len(have)}/{len(keys)} attached "
              f"({sorted(set(keys) - set(have))} missing) -- cannot check")
        ok = False
        continue
    f = ancova(D[D.run.isin(keys)], "run")
    exp_b, (exp_lo, exp_hi) = EXPECT[n]
    got_b = f["beta"][1]
    lo, hi = f["ci"][1]
    tol = 2e-3
    good = abs(got_b - exp_b) < tol and abs(lo - exp_lo) < 0.01 and abs(hi - exp_hi) < 0.01
    ok &= good
    show(f, f"{n} conditions, clustered by condition")
    print(f"    paper reports beta1 {exp_b:+.4f} CI [{exp_lo:+.3f}, {exp_hi:+.3f}]  "
          f"-> {'MATCHES' if good else '*** MISMATCH ***'}")

if not ok:
    print("\n  *** Reproduction incomplete. Treat everything below as provisional. ***")
    print("  Two different causes, and they need different responses:")
    print("    - a condition is simply NOT ATTACHED -> attach it, nothing is wrong;")
    print("    - all six are attached and beta1 still differs -> the inputs are not")
    print("      the ones the paper was written from. Find out which before editing")
    print("      a single number in the manuscript.")

print("\n" + "=" * 74)
print("THE NEW FIT  --  every run, clustered by RUN")
print("=" * 74)
print("  Once a condition has been trained more than once, the condition is no")
print("  longer the independent unit; the training RUN is. Clustering by condition")
print("  while pooling replicates inside it would treat two draws as one and")
print("  understate the standard error. This is the correctly specified version.")

_nlab, _ntr = D.run.nunique(), D.trun.nunique()
print(f"  {_nlab} condition labels come from {_ntr} distinct training runs.")
if _nlab != _ntr:
    _multi = (RM.groupby("trun").run.apply(list)
                .loc[lambda x: x.map(len) > 1].to_dict())
    for _t, _c in _multi.items():
        print(f"    {_t}: {len(_c)} checkpoints of ONE run -> {_c}")
    print("  Those are not independent clusters. The primary fit clusters on the")
    print("  TRAINING RUN; the by-label fit is shown after it for comparison only.")

BYLABEL = ancova(D, "run")
ALLRUNS = ancova(D, "trun")

show(ALLRUNS, f"PRIMARY: all runs, clustered by TRAINING RUN (G={_ntr})")
b1, (lo1, hi1) = ALLRUNS["beta"][1], ALLRUNS["ci"][1]
print(f"\n    per-decade multiplier on eps50: {10 ** b1:.3f}x  "
      f"[{10 ** lo1:.3f}, {10 ** hi1:.3f}]")
print(f"    interval {'EXCLUDES' if lo1 > 0 or hi1 < 0 else 'INCLUDES'} zero")

if _nlab != _ntr:
    show(BYLABEL, f"for comparison only: clustered by condition label (G={_nlab})")
    _l2 = BYLABEL["ci"][1]
    print(f"\n    That fit treats {_nlab - _ntr} checkpoint(s) of an existing run as")
    print(f"    independent draws, so its interval [{_l2[0]:+.4f}, {_l2[1]:+.4f}] is")
    print("    too narrow. Do not report it as the estimate.")

# beta1 is identified only by variation ACROSS exposure levels, so a reader may
# prefer a smaller df than G-1. Report the sensitivity rather than hide it.
_nx = D.hours.nunique()
_ngrp = RM.loc[RM.group.notna(), "group"].nunique()
print(f"\n  df sensitivity on beta1 (point estimate {b1:+.4f}, "
      f"clustered SE {ALLRUNS['se'][1]:.4f}):")
for _df, _lab in sorted({_ntr - 1: f"G-1, training runs ({_ntr}) [PRIMARY]",
                         _nx - 1: f"distinct exposure levels ({_nx}) - 1",
                         _ngrp - 1: f"replicate groups ({_ngrp}) - 1"}.items(),
                        reverse=True):
    if _df < 1: continue
    _t = t_crit(_df)
    _lo = b1 - _t * ALLRUNS["se"][1]; _hi = b1 + _t * ALLRUNS["se"][1]
    print(f"    df {_df:2d}  t={_t:.3f}  [{_lo:+.4f}, {_hi:+.4f}]  "
          f"{'excludes' if _lo > 0 else 'INCLUDES'} zero   {_lab}")

fits = {"all_by_training_run": ALLRUNS, "by_condition_label": BYLABEL}

## 6 · Between-run variance

Pooled within replicate groups. Milestones are excluded — they are not second runs of anything.

In [ ]:
print("\n" + "=" * 74)
print("BETWEEN-RUN VARIANCE IN ATTACK SUCCESS")
print("=" * 74)

CHI2 = {1: (0.000982, 5.0239), 2: (0.05064, 7.3778), 3: (0.21580, 9.3484),
        4: (0.48442, 11.1433), 5: (0.83121, 12.8325), 6: (1.23734, 14.4494),
        7: (1.68987, 16.0128), 8: (2.17973, 17.5345), 9: (2.70039, 19.0228),
        10: (3.24697, 20.4832), 11: (3.81575, 21.9200), 12: (4.40379, 23.3367)}

def chi2_bounds(df):
    if df in CHI2:
        return CHI2[df]
    try:
        from scipy.stats import chi2
        return float(chi2.ppf(0.025, df)), float(chi2.ppf(0.975, df))
    except Exception:
        return None, None

# Pool only within a REPLICATE GROUP (see the note beside PUBLISHED). Milestones
# carry group=None and are excluded: they are not second runs of anything.
by_grp, rows_v = {}, []
excluded = []
for r in RM.itertuples():
    if r.group is None or (isinstance(r.group, float) and np.isnan(r.group)):
        excluded.append(r.run); continue
    # (group, env): two runs are clean replicates of each other only if they
    # share BOTH the condition and the software they were trained with.
    by_grp.setdefault((r.group, r.env), []).append(r)
if excluded:
    print(f"  excluded from the variance estimate (not replicates of anything): "
          f"{excluded}")

num = den = 0.0
for (grp, env), rs in sorted(by_grp.items(), key=lambda kv: kv[1][0].hours):
    labels = [r.run for r in rs]
    asrs = [r.asr for r in rs]
    h = rs[0].hours
    # A replicate group whose members disagree on exposure is a construction
    # error, not a measurement: it would mean two "replicates" trained on
    # different amounts of audio.
    assert len({round(x.hours, 2) for x in rs}) == 1, (
        f"replicate group {grp} spans several exposures: "
        f"{sorted({(x.run, x.hours) for x in rs})}")
    if len(rs) < 2:
        print(f"  {grp:16s} {env:10s} ({h:6.2f} h): 1 run  ({labels[0]})  "
              f"ASR {asrs[0]:.1f}% -- no spread")
        continue
    sd = float(np.std(asrs, ddof=1))
    num += (len(rs) - 1) * sd ** 2
    den += len(rs) - 1
    rows_v.append(dict(group=grp, env=env, hours=h, rung=rs[0].rung, n_runs=len(rs),
                       sd_pp=round(sd, 2), asr_min=round(min(asrs), 1),
                       asr_max=round(max(asrs), 1), runs="; ".join(labels)))
    print(f"  {grp:16s} {env:10s} ({h:6.2f} h): {len(rs)} runs  "
          f"ASR {min(asrs):5.1f}-{max(asrs):5.1f}%  sd {sd:5.1f} pp   "
          f"[{', '.join(labels)}]")

sigma = sigma_raw = s_lo = s_hi = None
if den <= 0:
    print("\n  NO EXPOSURE LEVEL HAS TWO RUNS.")
    print("  Sigma is not estimable and the power table below cannot be produced.")
    print("  This is what NB-14 and NB-16 exist to fix.")
else:
    sigma_raw = (num / den) ** 0.5
    dfv = int(den)

    # ---------------------------------------------------------------------
    # BIAS CORRECTION, and why the paper's number differs from the raw one.
    #
    # The sample standard deviation underestimates sigma badly at tiny n:
    # E[s] = c4(df) * sigma, and c4(1) = 0.7979. The paper's sec:power quotes
    # ~30 pp from TWO runs (75.0% and 41.0%) by inverting the expected RANGE,
    # E[range] = 1.128 sigma for n=2 -- which is the same bias correction
    # arrived at from the other direction: 34/1.128 = 30.1, and
    # (34/sqrt(2))/0.7979 = 30.1 as well.
    #
    # The raw pooled sd is therefore NOT the number the paper reports, and
    # printing only one of them would make the manuscript look inconsistent
    # with its own notebook. Both are printed, and the correction shrinks
    # towards 1 as replicates arrive: c4(1)=0.798, c4(3)=0.921, c4(5)=0.952,
    # c4(10)=0.975.
    # ---------------------------------------------------------------------
    def c4(df):
        """E[s]/sigma for a chi-distributed s with `df` degrees of freedom."""
        from math import lgamma, exp, log
        return exp(0.5 * log(2.0 / df) + lgamma((df + 1) / 2.0) - lgamma(df / 2.0))

    c = c4(dfv)
    sigma = sigma_raw / c
    lo_q, hi_q = chi2_bounds(dfv)
    print(f"\n  pooled sd (raw)          = {sigma_raw:.2f} pp  (df {dfv}, "
          f"{len(rows_v)} within-environment replicate group(s))")
    print(f"  bias-corrected sigma     = {sigma:.2f} pp  (divided by c4({dfv}) = {c:.4f})")
    print("  The corrected value is the one to report and the one the paper's")
    print("  sec:power quotes. The correction matters at df 1-3 and is <5% by df 5.")
    if lo_q:
        # The chi-square interval is for sigma itself, so it is built from the
        # raw sd and needs no c4 factor.
        s_lo = sigma_raw * (dfv / hi_q) ** 0.5
        s_hi = sigma_raw * (dfv / lo_q) ** 0.5
        print(f"  95% CI for sigma = [{s_lo:.2f}, {s_hi:.2f}] pp")
        print("\n  Report the interval, not the point estimate. With this few runs sigma")
        print("  is itself a noisy statistic, and a bare point estimate invites a")
        print("  reviewer to ask how precise it is -- correctly.")
    print("\n  Against the pair effects it has to explain:")
    for rung, (ex, eff) in PAIR_EFFECTS.items():
        print(f"    rung {rung:3d} h ({ex:>3s}): effect {eff:4.1f} pp  "
              f"sigma/effect {sigma / eff:.2f}")
    pd.DataFrame(rows_v).to_csv(f"{WORK}/variance_by_group.csv", index=False)

    # ---- the confounded alternative, reported for completeness ------------
    # Pooling across environments adds the published runs back in. It gains df,
    # but every pair it adds differs in library version as well as in run, so it
    # is NOT the number to lead with. Printed so the difference is visible rather
    # than a judgement the reader has to take on trust.
    ng = nd = 0.0
    mixed = []
    for grp in sorted({g for g, _ in by_grp}):
        rs = [r for (g, e), v in by_grp.items() if g == grp for r in v]
        if len(rs) < 2:
            continue
        a = [r.asr for r in rs]
        envs = {r.env for r in rs}
        ng += (len(rs) - 1) * float(np.var(a, ddof=1))
        nd += len(rs) - 1
        mixed.append((grp, len(rs), len(envs)))
    if nd > den:
        s_mix = (ng / nd) ** 0.5 / c4(int(nd))
        print(f"\n  CONFOUNDED alternative, pooling across environments: "
              f"sigma = {s_mix:.2f} pp (df {int(nd)})")
        for grp, n_r, n_e in mixed:
            if n_e > 1:
                print(f"    {grp}: {n_r} runs spanning {n_e} library versions")
        print("  Do not lead with this. It mixes run-to-run variance with the")
        print("  4.44.2-to-5.16.1 difference, which is a separate question.")
        print(f"  If it is close to {sigma:.1f} pp, that is mild evidence the library")
        print("  version does not matter much here -- worth one sentence, not a table.")
    pd.DataFrame([dict(sigma_raw_pp=round(sigma_raw, 3), c4=round(c, 5),
                       sigma_pp=round(sigma, 3), df=dfv,
                       sigma_lo=round(s_lo, 3) if s_lo else None,
                       sigma_hi=round(s_hi, 3) if s_hi else None,
                       n_groups=len(rows_v))]
                ).to_csv(f"{WORK}/sigma.csv", index=False)

## 7 · Runs per condition

In [ ]:
print("\n" + "=" * 74)
print("RUNS PER CONDITION REQUIRED")
print("=" * 74)
if sigma is None:
    print("  Not computable without sigma.")
    POWER_ROWS = []
else:
    # n per group = 2 (z_{a/2} + z_b)^2 sigma^2 / delta^2, normal approximation.
    # z_{0.975}=1.959964, z_{0.80}=0.841621 -> 2*(2.801585)^2 = 15.6978
    K_POW = 2 * (1.959964 + 0.841621) ** 2
    print(f"  Two-sided alpha={ALPHA}, power={POWER:.0%}, two independent groups of runs.")
    print(f"  n per condition = {K_POW:.3f} * sigma^2 / delta^2\n")
    POWER_ROWS = []
    for rung, (ex, eff) in PAIR_EFFECTS.items():
        n_pt = K_POW * sigma ** 2 / eff ** 2
        n_lo = K_POW * s_lo ** 2 / eff ** 2 if s_lo else None
        n_hi = K_POW * s_hi ** 2 / eff ** 2 if s_hi else None
        # The formula is continuous and can return less than 2. Two runs per group
        # is the floor of the design -- you cannot contrast fewer -- so report
        # max(2, ceil(n)) as the requirement and keep the raw value in the CSV.
        n_use = max(2, int(math.ceil(n_pt)))
        POWER_ROWS.append(dict(rung=rung, exposure_ratio=ex, effect_pp=eff,
                               runs_needed=n_use, runs_raw=round(n_pt, 2),
                               runs_lo=round(n_lo, 1) if n_lo else None,
                               runs_hi=round(n_hi, 1) if n_hi else None))
        s = (f"  detect {eff:4.1f} pp (rung {rung:3d} h): {n_use:3d} runs per condition"
             f"  (raw {n_pt:.2f}")
        s += f", 95% CI {n_lo:.1f}-{n_hi:.1f})" if n_lo else ")"
        print(s)
    print("\n  Smallest detectable effect at a given number of runs per condition:")
    for n in (2, 3, 5, 10):
        print(f"    {n:2d} runs: {sigma * (K_POW / n) ** 0.5:5.1f} pp")
    print("\n  Total fine-tuning runs for the full design, at the largest requirement:")
    drive = max(POWER_ROWS, key=lambda r: r["runs_needed"])
    w = int(drive["runs_needed"])
    print(f"    Driven by the SMALLEST effect ({drive['effect_pp']} pp, the "
          f"{drive['exposure_ratio']} pair): {w} runs per condition.")
    print(f"    {w} x 7 conditions = {w * 7} runs, ~{w * 7 * 3.5:.0f} GPU-h of "
          f"training plus ~{w * 7 * 4.6:.0f} GPU-h of attacks = "
          f"~{w * 7 * 8.1:.0f} GPU-h.")
    print(f"    At Kaggle's 30 GPU-h/week that is ~{w * 7 * 8.1 / 30:.0f} weeks of "
          "quota, which is why the paper calls it a different project.")
    pd.DataFrame(POWER_ROWS).to_csv(f"{WORK}/power.csv", index=False)

## 8 · Write the tables

In [ ]:
RM.to_csv(f"{WORK}/runs.csv", index=False)

tex = []
tex.append("% --- Table: every run, grouped by exposure ---")
tex.append(r"\begin{tabular}{llrrrr}")
tex.append(r"\toprule")
tex.append(r"Run & env & exposure (h) & clean WER & ASR & $n$ \\")
tex.append(r"\midrule")
for r in RM.sort_values(["hours", "run"]).itertuples():
    tex.append(f"{r.run} & {r.env} & {r.hours:.2f} & {r.wer:.4f} & "
               f"{r.asr:.1f}\\% & {r.n} \\\\")
tex.append(r"\bottomrule")
tex.append(r"\end{tabular}")
if sigma is not None:
    tex.append("")
    tex.append(f"% between-run sigma = {sigma:.2f} pp (bias-corrected, c4), "
               f"raw pooled sd {sigma_raw:.2f}, df {int(den)}"
               + (f", 95% CI [{s_lo:.2f}, {s_hi:.2f}]" if s_lo else ""))
b = ALLRUNS
tex.append("")
tex.append(f"% ANCOVA clustered on the TRAINING RUN: beta1 = {b['beta'][1]:+.4f} "
           f"CI [{b['ci'][1][0]:+.4f}, {b['ci'][1][1]:+.4f}] "
           f"(G={b['G']} training runs, N={b['N']}, df={b['df']})")
open(f"{WORK}/tables.tex", "w").write("\n".join(tex) + "\n")

L = [f"# {time.strftime('%Y-%m-%d')} — NB-17 variance components and run-clustered ANCOVA",
     "",
     f"{len(RM)} runs, {len(D)} utterance rows. Reproduction check: "
     f"{'PASSED' if ok else 'INCOMPLETE'}.", ""]
L.append("| run | env | exposure h | clean WER | ASR | n | source |")
L.append("|---|---|---:|---:|---:|---:|---|")
for r in RM.sort_values(["hours", "run"]).itertuples():
    L.append(f"| {r.run} | {r.env} | {r.hours:.2f} | {r.wer:.4f} | {r.asr:.1f}% | "
             f"{r.n} | {r.source} |")
L += ["", f"**ANCOVA clustered on the training run:** beta1 = {b['beta'][1]:+.4f}, "
          f"95% CI [{b['ci'][1][0]:+.4f}, {b['ci'][1][1]:+.4f}], "
          f"G={b['G']}, N={b['N']}, t df={b['df']}. "
          f"Per decade: {10 ** b['beta'][1]:.3f}x. Interval "
          f"{'excludes' if b['ci'][1][0] > 0 or b['ci'][1][1] < 0 else 'includes'} zero."]
if sigma is not None:
    L.append(f"**Between-run sigma = {sigma:.2f} pp** (bias-corrected; raw pooled sd "
             f"{sigma_raw:.2f}, df {int(den)}"
             + (f", 95% CI [{s_lo:.2f}, {s_hi:.2f}]" if s_lo else "") + ").")
    for r in POWER_ROWS:
        L.append(f"* detect {r['effect_pp']} pp -> {r['runs_needed']} runs per condition"
                 + (f" (CI {r['runs_lo']}-{r['runs_hi']})" if r['runs_lo'] else ""))
open(f"{WORK}/RUNLOG_BLOCK.md", "w").write("\n".join(L) + "\n")

json.dump(dict(notebook="NB-17-variance-components", n_runs=len(RM), n_rows=len(D),
               reproduction_ok=bool(ok), sigma_pp=sigma,
               sigma_raw_pp=sigma_raw if sigma is not None else None,
               sigma_ci=[s_lo, s_hi] if s_lo else None,
               beta1=float(b["beta"][1]), beta1_ci=[float(x) for x in b["ci"][1]],
               G=int(b["G"]), N=int(b["N"]), df=int(b["df"]),
               runs=RM.to_dict("records"),
               timestamp=time.strftime("%Y-%m-%d %H:%M:%S")),
          open(f"{WORK}/_provenance.json", "w"), indent=2, default=str)
json.dump({"title": "variance-components", "id": "YOURUSERNAME/variance-components",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)

print("\n" + "=" * 74)
print("FILES WRITTEN")
print("=" * 74)
for f in sorted(os.listdir(WORK)):
    if f.endswith((".csv", ".tex", ".md", ".json")):
        print("   ", f)
print("\n" + open(f"{WORK}/RUNLOG_BLOCK.md").read())

---

## What goes into the paper

| output | where it goes |
|---|---|
| `sigma.csv` | `sec:power` — replaces the $n=2$ extrapolation with a measurement |
| `variance_by_group.csv` | a new table in `sec:replication`: sigma at each rung |
| `power.csv` | `sec:power` — runs per condition, with the interval |
| `runs.csv`, `tables.tex` | extends `tab:eps50` from 7 rows to every run |
| run-clustered beta1 | `tab:ancova` gains a third row, and it becomes the primary fit |

## The one decision this notebook cannot make for you

If the run-clustered interval **excludes** zero once the replicates are in, the paper's
headline changes: there would be an identifiable data-volume effect after all, measured against
a properly estimated between-run variance. That is a different and better paper, and it would
need the abstract, introduction and conclusion rewritten rather than patched.

If it still **includes** zero, the current framing stands, now backed by sigma at three rungs
instead of one pair.

Either way the pre-specified fit stays in the manuscript, and so does the reason: changing
which analysis is called primary after seeing its result is the practice these controls exist
to prevent. The run-clustered fit is reported as what it is — a correction to the clustering
level forced by the arrival of replicates, and specified before those replicates were attacked.
